# Isolating a Fortran Child Procedure with Complex Dependencies

This notebook demonstrates how to isolate a single Fortran procedure — a
subroutine or a function — from a larger codebase, resolving every dependency
it carries with it, and producing a self-contained compilation unit that can
be built and run on its own.

## Why a *child* procedure?

The examples we have seen so far involve a main program that
calls a handful of subroutines, each of which may itself call other routines
and reference module-level state. The interesting case is the **child**: a
procedure that is *not* the entry point of the program, that lives inside a
module or a `contains` block, and that depends on

- dummy arguments (its own inputs and outputs),
- local variables declared in its specification part,
- module-level variables it accesses through `USE`,
- **other procedures** — subroutines and functions — that it calls,
- **intrinsic functions** and operators,
- and, occasionally, **initialised parameters** whose right-hand side
  references yet other names (`real(dp), parameter :: dt = dt_max`).

A child procedure therefore has a **dependency graph** of its own, which may
be deeper and wider than the procedure's own body suggests. Isolating the
child means walking that graph, pulling in every node it transitively depends
on, and emitting a new compilation unit that reproduces the child's behaviour
exactly, without the rest of the original program.

## What "complex" means here

The dependency of a child procedure can be called *complex* when it involves
any of the following:

- **Multiple levels of call nesting.** The child calls a helper, which calls
  another helper, which uses a module variable. The isolator must recurse
  until no unresolved names remain.
- **Cross-module references.** The child uses a variable defined in a module
  that also contains other, unrelated routines. Only the parts actually
  reached by the child should be pulled in.
- **Parameter initialisation.** A local parameter may be declared as
  `real(dp), parameter :: dt = dt_max`, where `dt_max` is a module constant.
  The name `dt_max` appears nowhere in the child's executable part — it lives
  only in the right-hand side of an initialisation. The isolator must detect
  this and treat `dt_max` as a global dependency.
- **Assumed-length character arguments.** `character(len=*), intent(in) :: filename`
  has no explicit length, so the isolator must invent a mechanism (a companion
  integer argument carrying the length, plus `character(len=:), allocatable`
  in the isolated version) to preserve semantics.
- **Implicitly shaped arrays.** A dummy argument declared `real(dp) :: arr(:)`
  has no shape until the caller supplies one. The isolator must resolve the
  shape from the calling context.
- **Intrinsic-name collisions.** A local variable named `size` or `sum` shadows
  an intrinsic of the same name; the isolator must not confuse the two.

## The approach, bottom to top

Isolation is performed **bottom-up** along the call graph:

1. Start with the deepest child — the one whose own dependency graph is
   smallest, ideally a pure utility with no calls of its own.
2. Emit its isolated form, together with any module variables it reads and any
   module procedures it references.
3. Move one level up: isolate its parent, using the already-isolated children
   as building blocks.
4. Continue until the top-level procedure of interest has been isolated.

This ordering guarantees that when a procedure is isolated, everything it
depends on already exists in isolated form. The result is a chain of
self-contained compilation units that can be linked together or, if only the
top-level procedure is wanted, folded into a single file.

## What we will work through

The rest of this notebook applies the method to the multi-module EBM example:

- we pick a **leaf** procedure first (`mean`, `clamp`, `solve_tridiag`),
- then a procedure that depends on leaves (`surface_albedo`, `diagnose`),
- then a procedure that depends on both leaves and mid-level helpers
  (`step`, `report`),
- and finally the driver (`main_program`), showing how the isolated chain
  reproduces the original output to within `TOL`.

At each stage the notebook reports:

- the list of **dummy**, **local**, and **global** names discovered,
- the **call graph** extracted from the procedure's execution part,
- the **initialisation dependencies** detected in the specification part,
- and the resulting **self-contained source**, ready to compile.

In [1]:
# Notebook Setup

%load_ext autoreload
%autoreload 2

# Change to the Fgpt directory
%cd /home/kardaneh/Fgpt

# Necessay imports 
import sys
from fgpt import Isolator
from fgpt.isolator import parse_args

/home/kardaneh/Fgpt/.venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/home/kardaneh/Fgpt


In [2]:
sys.argv = [
    "my_program",
    "--rest_of_path",
    "Fgpt/examples/dependencies",
    "--target_module",
    "main_program",
    "--work",
    "/home/kardaneh",
    "--target_model",
    "general",
    "--parent_subroutine",
    "main_program",
    "--target_subroutines",
    "main_program",
    "--openacc",
    "False",
    "--f2py",
    "False",
    "--py2jx",
    "False",
    "--tapenade",
    "False",
    "--mode",
    "jax",
    "--benchmark_dir",
    "/tmp/benchmark",
    "--config_path",
    "/home/kardaneh/Fgpt/template.yaml",
    "--vectorize",
    "kjpindex",
    "nvm",
    "npts",
]

In [3]:
args = parse_args()

isolator = Isolator(
    rest_of_path=args.rest_of_path,
    target_model=args.target_model,
    target_module=args.target_module,
    work=args.work,
    config_path=args.config_path,
    openacc=args.openacc,
    tapenade=args.tapenade,
    f2py=args.f2py,
    py2jx=args.py2jx,
)

isolator.run(
    benchmark_dir=args.benchmark_dir,
    vectorize=args.vectorize,
    mode=args.mode,
    parent_subroutine=args.parent_subroutine,
    target_subroutines=args.target_subroutines,
)
    

╭────────────────────────────────── Fortran General purpose Transformer (Fgpt) ───────────────────────────────────╮
│ 🚀 Starting Module: Isolator                                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[INFO] Successfully parsed file: /home/kardaneh/Fgpt/examples/dependencies/main_program.f90

[INFO] Normalizing names in parsed file: /home/kardaneh/Fgpt/examples/dependencies/main_program.f90

[INFO] Successfully normalized all names in the module

[INFO] Starting isolation for target subroutines: ['main_program'] from parent subroutine: main_program

...

[INFO] Benchmark files exist. Now running the unit tests ...

 
  1-D Energy Balance Model
 
  Grid size              :            32
  Lat min / max          :    -87.18750000000000         87.18750000000000     
  Sum of area weights    :     1.000000000000000     
  Mean insolation (W/m^2):     340.1842530703841     
  linspace mean      :     1.500000000000000     
  linspace variance  :    8.8709677419354857E-002
  linspace stddev    :    0.2978416985906353     
  linspace median    :     1.500000000000000     
  corr(x, x)         :     1.000000000000000     
 
 
  Step                      :            25
  Mean T (K)                :     100.0000000000000     
  Std dev (K)               :     0.000000000000000     
  Median (K)                :     100.0000000000000     
  Area-weighted mean T (K)  :     99.99999999999997     
  Pole-to-pole gradient (K) :     0.000000000000000     
  corr(insolation, T)       :     0.000000000000000     
  TOA imbalance (W/m^2)     :    -233.7106813686279     
  Albedo min / max          :    0.30000000

[INFO] Execution completed in /home/kardaneh/Fgpt/main_program/main_program

[INFO] Successfully wrote code to file: /home/kardaneh/Fgpt/examples/dependencies/main_program.f90

╭─────────────────────────────────────────────── ✅ Node Complete ────────────────────────────────────────────────╮
│ Exiting node: Isolated Procedure                                                                                │
│ Duration: 28.21s                                                                                                │
│ ✅ Done isolating → main_program                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯